# Анализ эксперимента по определению setpoint от 16 января 2026 г

In [ ]:
import matplotlib.pyplot as plt

from pathlib import Path
from nn_laser_stabilizer.config.config import load_config

EXPERIMENT_DIR = "2026-01-16_20-32-47_setpoint_determination" # 2026-01-16_20-36-02
EXPERIMENT_DIR_PATH = Path(f"../experiments/{EXPERIMENT_DIR}")

ENV_LOG_PATH = EXPERIMENT_DIR_PATH / "connection.log"

CONFIG_PATH = EXPERIMENT_DIR_PATH / "config.yaml"
config = load_config(CONFIG_PATH)
print(f"Эксперимент: {config.experiment_name}")

In [ ]:
import pandas as pd

from nn_laser_stabilizer.analysis.sources import parse_keyval_log

# connection.log: обмен с PID-контроллером ([PID] send/read).
# i-я посылка (send) спаривается с i-м чтением (read).
_raw = parse_keyval_log(ENV_LOG_PATH)
_pid = _raw[_raw["tag"] == "PID"]

_send = _pid[_pid["event"] == "send"][
    ["kp", "ki", "kd", "control_min", "control_max", "setpoint"]
].reset_index(drop=True)
_send["_i"] = range(len(_send))
_read = _pid[_pid["event"] == "read"][
    ["process_variable", "control_output"]
].reset_index(drop=True)
_read["_i"] = range(len(_read))
_pairs = _read.merge(_send, on="_i", how="left")

connection_df = pd.DataFrame({
    "Connection step": _pairs["_i"],
    "Type": "exchange",
    "Kp": _pairs["kp"],
    "Ki": _pairs["ki"],
    "Kd": _pairs["kd"],
    "Control min": _pairs["control_min"],
    "Control max": _pairs["control_max"],
    "Process variable": _pairs["process_variable"],
    "Control output": _pairs["control_output"],
    "Setpoint": _pairs["setpoint"],
})
print(f"Загружено {len(connection_df)} записей")

In [ ]:
plt.figure(figsize=(12, 5))
plt.plot(connection_df['Connection step'], connection_df['Process variable'], 'b-', alpha=0.7, linewidth=0.8, label='Process Variable')

plt.title('Process Variable')
plt.xlabel('Step')
plt.ylabel('Process Variable')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

plt.figure(figsize=(12, 5))
plt.plot(connection_df['Connection step'], connection_df['Control output'], 'g-', alpha=0.7, linewidth=0.8, label='Control Output')

plt.title('Control Output')
plt.xlabel('Step')
plt.ylabel('Control Output')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()